# CXR-DetectBench Phase 2 - Data Preprocessing

Execute Phase 2 preprocessing pipeline:
- Multi-annotator fusion (raw / wbf / nms)
- Fusion ablation with YOLOv8n
- COCO to YOLO format conversion

## 1. Environment Setup

In [ ]:
!pip install -q ensemble-boxes pycocotools opencv-python-headless ultralytics scikit-learn tqdm matplotlib

In [ ]:
import os
from pathlib import Path

os.chdir('/kaggle/working')

print("=== Available input directories ===")
!ls -la /kaggle/input/

## 2. Explore Input Paths

Kaggle mounts competition data and datasets differently. Let's find the actual file locations.

In [ ]:
INPUT_BASE = Path('/kaggle/input')

# Find PNG dataset path (corochann/vinbigdata-chest-xray-original-png)
png_dirs = list(INPUT_BASE.glob('*chest-xray-original*')) + list(INPUT_BASE.glob('*png*'))
print(f"PNG candidates: {[str(d) for d in png_dirs]}")

# Find competition data (vinbigdata-chest-xray-abnormalities-detection)
comp_dirs = list(INPUT_BASE.glob('*abnormalities*')) + list(INPUT_BASE.glob('*vinbigdata*'))
print(f"Competition candidates: {[str(d) for d in comp_dirs]}")

# List ALL input directories
for d in sorted(INPUT_BASE.iterdir()):
    if d.is_dir():
        files = list(d.iterdir())[:20]
        print(f"\n{d.name}/ ({len(list(d.iterdir()))} items):")
        for f in sorted(files)[:15]:
            print(f"  {f.name}")

## 3. Clone Repository

In [ ]:
!git clone https://github.com/kinjazA/cxr-detectbench.git
%cd cxr-detectbench

## 4. Prepare Data Files

Copy train.csv and images.csv from input sources to local working directory.

In [ ]:
!mkdir -p data/raw data/processed

# --- Find and copy train.csv ---
# Competition data: check multiple possible paths
import subprocess

train_csv_found = False
train_csv_paths = [
    '/kaggle/input/vinbigdata-chest-xray-abnormalities-detection/train.csv',
    '/kaggle/input/vinbigdata-chest-xray-abnormalities-detection/train/train.csv',
]

# Also search using glob
for path in Path('/kaggle/input').rglob('train.csv'):
    train_csv_paths.append(str(path))

for src in train_csv_paths:
    if Path(src).exists():
        print(f"Found train.csv at: {src}")
        !cp '{src}' data/raw/train.csv
        train_csv_found = True
        break

if not train_csv_found:
    print("ERROR: train.csv not found!")
    print("\nDownloading from Kaggle API...")
    import subprocess
    result = subprocess.run(
        ['kaggle', 'competitions', 'download', 'vinbigdata-chest-xray-abnormalities-detection', '-f', 'train.csv', '-p', 'data/raw'],
        capture_output=True, text=True)
    print(result.stdout, result.stderr)

# --- Find and copy images.csv (with Rows/Columns for image size) ---
images_found = False

# Search for train_meta.csv or images.csv
meta_names = ['train_meta.csv', 'images.csv', 'image_meta.csv']
for name in meta_names:
    for path in Path('/kaggle/input').rglob(name):
        if path.exists():
            print(f"Found {name} at: {path}")
            !cp '{path}' data/raw/images.csv
            images_found = True
            break
    if images_found:
        break

if not images_found:
    print("WARNING: images.csv not found! label_fusion.py will fail.")
    print("Make sure 'sunhwan/vinbigdata-chest-xray-dicom-metadata' is added as a Dataset source in Kaggle.")

# --- Link images ---
!ln -sf /kaggle/input/vinbigdata-chest-xray-original-png/train data/processed/images_png 2>/dev/null || true

print("\n=== data/raw/ contents ===")
!ls -lh data/raw/
print("\n=== data/processed/ contents ===")
!ls -lh data/processed/

## 5. Multi-Annotator Fusion

Generate 3 COCO annotation variants: raw / wbf / nms

In [ ]:
# Verify required input files exist
import sys
required = ['data/raw/train.csv', 'data/raw/images.csv']
for f in required:
    if not Path(f).exists():
        print(f"ERROR: {f} not found! Check previous cell.")
        sys.exit(1)
    else:
        print(f"OK: {f}")

# Run fusion
!python scripts/label_fusion.py \
    --train_csv data/raw/train.csv \
    --images_csv data/raw/images.csv \
    --output_dir data/processed/labels_coco \
    --iou_thr 0.5

In [ ]:
# Check outputs
!python scripts/check_fusion_output.py

## 6. Fusion Ablation Experiment

Quick YOLOv8n training (20 epochs) on 3 fusion variants to select best strategy.

In [ ]:
# Verify fusion outputs exist before running ablation
for mode in ['raw', 'wbf', 'nms']:
    p = Path(f'data/processed/labels_coco/{mode}/annotations.json')
    if not p.exists():
        print(f"ERROR: {p} not found!")
        sys.exit(1)

print("All 3 COCO variants ready. Starting ablation...")

!python scripts/run_fusion_ablation.py \
    --coco_dir data/processed/labels_coco \
    --images_dir data/processed/images_png \
    --train_csv data/raw/train.csv \
    --epochs 20 \
    --imgsz 640 \
    --batch 16 \
    --output phase2_fusion_ablation.csv

In [ ]:
# Display results
import pandas as pd

result_path = Path('phase2_fusion_ablation.csv')
if result_path.exists():
    results = pd.read_csv(result_path)
    print("\n" + "="*60)
    print("Phase 2.4 Fusion Strategy Ablation Results")
    print("="*60)
    print(results.to_string(index=False))
    
    best_mode = results.loc[results['mAP50'].idxmax(), 'fusion_mode']
    best_map = results.loc[results['mAP50'].idxmax(), 'mAP50']
    print(f"\nBest fusion strategy: {best_mode} (mAP@0.5={best_map:.4f})")
else:
    print("Ablation results not yet available.")

## 7. Generate Final YOLO Labels

Convert best fusion variant to YOLO format for use in Phase 3-5.

In [ ]:
# Set this to the best fusion mode from ablation results above
BEST_FUSION = 'wbf'  # Update after checking ablation results

print(f"Generating YOLO labels for fusion mode: {BEST_FUSION}")

!python scripts/convert_coco_yolo.py \
    --coco_json data/processed/labels_coco/{BEST_FUSION}/annotations.json \
    --output_dir data/processed/labels_yolo/{BEST_FUSION}

print(f"\nYOLO labels generated: data/processed/labels_yolo/{BEST_FUSION}/")

## 8. Verify Outputs

In [ ]:
print("=== Phase 2 Outputs ===")

print("\n1. COCO annotations:")
!ls -lh data/processed/labels_coco/*/annotations.json 2>/dev/null || echo "Not found"

print("\n2. YOLO labels:")
!ls data/processed/labels_yolo/{BEST_FUSION}/ 2>/dev/null | head -10

print("\n3. Ablation results:")
!cat phase2_fusion_ablation.csv 2>/dev/null || echo "Not found"

print("\n4. Image link:")
!ls -la data/processed/images_png 2>/dev/null | head -5

print("\n=== Phase 2 Complete ===")
print("Remember to: Save Version -> Save & Run All (Commit)")
print("to persist outputs as a Kaggle Dataset for Phase 3-5.")